# 02 — Dense vs. Sparse Retrieval

*Level 2 — Advanced RAG*

## Objective
Measure dense (embedding) vs. sparse (BM25) retrieval quality against **real relevance judgments (qrels)** from BeIR/scifact — not a heuristic, unlike Level 1's evaluation set.


In [1]:
import sys
from pathlib import Path

LEVEL_DIR = Path.cwd().parent
sys.path.insert(0, str(LEVEL_DIR))
sys.path.insert(0, str(LEVEL_DIR / "hybrid-search"))
sys.path.insert(0, str(LEVEL_DIR / "query-transformations"))
sys.path.insert(0, str(LEVEL_DIR / "metadata-filtering"))
sys.path.insert(0, str(LEVEL_DIR / "context-compression"))


In [2]:
from common.dataset import prepare
from retrieval.dense import DenseRetriever
from retrieval.sparse import BM25Retriever
from retrieval.top_k_experiments import sweep_top_k

data = prepare()
corpus_texts = {d: data.corpus_text(d) for d in data.doc_ids()}
print(f"corpus={len(corpus_texts)} eval queries={len(data.queries)}")


corpus=1000 eval queries=300


In [3]:
dense = DenseRetriever.from_corpus(corpus_texts)   # cached after first run

sparse = BM25Retriever.from_corpus(corpus_texts)
print("retrievers ready")


retrievers ready


## Recall@K across the full 300-query test set


In [4]:
k_values = (1, 3, 5, 10, 20)
dense_recall = sweep_top_k(dense, data.queries, data.qrels, k_values)
sparse_recall = sweep_top_k(sparse, data.queries, data.qrels, k_values)

print(f"{'k':>4} {'dense':>8} {'sparse':>8}")
for k in k_values:
    print(f"{k:>4} {dense_recall[k]:>8.3f} {sparse_recall[k]:>8.3f}")


   k    dense   sparse
   1    0.733    0.683
   3    0.873    0.807
   5    0.900    0.847
  10    0.917    0.867
  20    0.943    0.890


## One query, two rankings


In [5]:
# Picked because dense finds the relevant doc in its Top-5 here while
# sparse (BM25) does not -- a genuine illustration of dense's strength
# on paraphrased/semantic queries.
sample_qid = "13"
sample_query = data.queries[sample_qid]
relevant = set(data.qrels[sample_qid])
print(f"Query: {sample_query!r}")
print(f"Known relevant doc(s): {relevant}\n")

print("Dense Top-5:")
for doc_id, score in dense.search(sample_query, top_k=5):
    mark = " <-- relevant" if doc_id in relevant else ""
    print(f"  {score:.3f}  {doc_id}{mark}  {data.corpus[doc_id]['title'][:70]}")

print("\nSparse (BM25) Top-5:")
for doc_id, score in sparse.search(sample_query, top_k=5):
    mark = " <-- relevant" if doc_id in relevant else ""
    print(f"  {score:.3f}  {doc_id}{mark}  {data.corpus[doc_id]['title'][:70]}")


Query: '5% of perinatal mortality is due to low birth weight.'
Known relevant doc(s): {'1606628'}

Dense Top-5:
  0.680  4791384  Neonatal Mortality Levels for 193 Countries in 2009 with Trends since 
  0.672  27123743  Role of birthweight in the etiology of breast cancer.
  0.662  7662395  Perinatal mortality in rural China: retrospective cohort study.
  0.640  2425364  Association between maternal serum 25-hydroxyvitamin D level and pregn
  0.630  1606628 <-- relevant  Estimates of global prevalence of childhood underweight in 1990 and 20

Sparse (BM25) Top-5:
  27.321  7662395  Perinatal mortality in rural China: retrospective cohort study.
  26.546  17450673  Intrauterine environments and breast cancer risk: meta-analysis and sy
  23.712  12779444  Effect of screening on cervical cancer mortality in England and Wales:
  22.214  19799455  Ascorbic-acid transporter Slc23a1 is essential for vitamin C transport
  21.384  2425364  Association between maternal serum 25-hydroxyvitamin D l

## What I observed

On this run, **dense retrieval beat sparse at every K** (e.g. Recall@1 ≈ 0.73 dense vs. 0.68 sparse; Recall@20 ≈ 0.94 vs. 0.89). That's a property of *this* corpus, not a universal rule: scifact's queries are claims that **paraphrase** the abstracts that support or refute them, rather than reusing their exact wording — exactly the case embeddings are good at and keyword matching is not.

BM25 would look very different on a corpus full of exact identifiers, part numbers, or rare technical terms an embedding model tends to blur together. This is *why* hybrid search exists — see the next notebook.

## Next

[03 — Hybrid Search & RRF](./03_hybrid_search_rrf.ipynb)
